# UNet Step 3 — Prototype 3D Visualisation

Shows **15 learned prototypes** (5 per class: NCR / ED / ET) from the `step3_proto_losses` run
(`base_channels=16`, `protos_per_class=5`).

**Two figures per run:**
1. **3-D contact sheet** — 8 evenly-spaced axial slices for every prototype,
   showing the similarity heatmap evolving across depth. The slice with peak
   activation gets a red border.
2. **Orthogonal view at peak** — axial / coronal / sagittal centred on the
   highest-activation voxel for each prototype.

**Before running:**
1. Add `AWS_ACCESS_KEY_ID` and `AWS_SECRET_ACCESS_KEY` to Colab Secrets (🔑 sidebar).
2. Set `JOB_NAME` (leave `None` to auto-pick the latest step-3 job).
3. Set `VOLUME_ID`.

In [ ]:
import tensorflow as tf
print(f'TensorFlow {tf.__version__}')
gpus = tf.config.list_physical_devices('GPU')
print(f'GPUs: {gpus}')
for g in gpus:
    tf.config.experimental.set_memory_growth(g, True)
if not gpus:
    print('⚠  No GPU — go to Runtime → Change runtime type → T4 GPU')

In [ ]:
%%capture
!pip install boto3 h5py scipy tqdm tf-keras -q

In [ ]:
import os
from google.colab import userdata

os.environ['AWS_ACCESS_KEY_ID']     = userdata.get('AWS_ACCESS_KEY_ID')
os.environ['AWS_SECRET_ACCESS_KEY'] = userdata.get('AWS_SECRET_ACCESS_KEY')
os.environ['AWS_DEFAULT_REGION']    = 'eu-central-1'

import boto3
boto3.client('sts').get_caller_identity()
print('AWS credentials OK')

In [ ]:
# ── Change these ──────────────────────────────────────────────────────────────
S3_BUCKET        = 'your-brats2020-data'
S3_DATA_PREFIX   = 'preprocessed_data_cropped'
S3_OUTPUT_PREFIX = 'step3-outputs'

JOB_NAME  = None   # e.g. 'unet-proto-losses-20260430-140000', or None to auto-pick
VOLUME_ID = 1      # which BraTS volume to visualise

# ── Architecture — must match training (do NOT change) ────────────────────────
BASE_CHANNELS    = 16
PROTOS_PER_CLASS = 5
N_CLASSES        = 4
NUM_SLICES       = 128
HEIGHT, WIDTH    = 192, 160

# ── Derived constants ─────────────────────────────────────────────────────────
NUM_PROTOTYPES = PROTOS_PER_CLASS * 3   # 15
PROTO_DIM      = BASE_CHANNELS * 8      # 128
POOL_FACTOR    = 8                      # 3 encoder stages of (1,2,2) pool
Hb = HEIGHT // POOL_FACTOR             # 24
Wb = WIDTH  // POOL_FACTOR             # 20

CLASS_NAMES  = {1: 'NCR', 2: 'ED', 3: 'ET'}
CLASS_COLORS = {1: '#e07b54', 2: '#5b9bd5', 3: '#70ad47'}

MODEL_DIR  = '/content/step3_model'
DATA_DIR   = '/content/brats_data'
OUTPUT_DIR = '/content/proto_vis'
for d in [MODEL_DIR, DATA_DIR, OUTPUT_DIR]:
    os.makedirs(d, exist_ok=True)

print(f'Prototypes: {NUM_PROTOTYPES} ({PROTOS_PER_CLASS}/class × 3 classes)  '
      f'proto_dim={PROTO_DIM}  bottleneck=({NUM_SLICES},{Hb},{Wb})')

In [ ]:
model_code = '''
import tensorflow as tf
import tf_keras as keras
from tf_keras import layers


class ConvBlock(keras.layers.Layer):
    def __init__(self, filters, **kw):
        super().__init__(**kw)
        self.conv1 = layers.Conv3D(filters, 3, padding="same", use_bias=False,
                                   kernel_initializer="he_normal")
        self.norm1 = layers.LayerNormalization()
        self.relu1 = layers.ReLU()
        self.conv2 = layers.Conv3D(filters, 3, padding="same", use_bias=False,
                                   kernel_initializer="he_normal")
        self.norm2 = layers.LayerNormalization()
        self.relu2 = layers.ReLU()

    def call(self, x):
        x = self.relu1(self.norm1(self.conv1(x)))
        x = self.relu2(self.norm2(self.conv2(x)))
        return x


class EncoderBlock(keras.layers.Layer):
    def __init__(self, filters, **kw):
        super().__init__(**kw)
        self.conv = ConvBlock(filters)
        self.pool = layers.MaxPool3D(pool_size=(1, 2, 2))

    def call(self, x):
        skip = self.conv(x)
        return skip, self.pool(skip)


class DecoderBlock(keras.layers.Layer):
    def __init__(self, filters, **kw):
        super().__init__(**kw)
        self.upsample = layers.Conv3DTranspose(
            filters, kernel_size=(1, 2, 2), strides=(1, 2, 2),
            padding="same", kernel_initializer="he_normal")
        self.concat = layers.Concatenate()
        self.conv   = ConvBlock(filters)

    def call(self, x, skip):
        x = self.upsample(x)
        x = self.concat([x, skip])
        return self.conv(x)


class UNet3DProto(keras.Model):
    def __init__(self, n_classes=4, base_channels=16, protos_per_class=5, **kw):
        super().__init__(**kw)
        c                     = base_channels
        self.protos_per_class = protos_per_class
        self.num_prototypes   = protos_per_class * 3
        self.proto_dim        = c * 8

        self.enc1       = EncoderBlock(c)
        self.enc2       = EncoderBlock(c * 2)
        self.enc3       = EncoderBlock(c * 4)
        self.bottleneck = ConvBlock(c * 8)
        self.dropout    = layers.Dropout(0.2)

        self.prototype_vectors = tf.Variable(
            tf.initializers.GlorotUniform()(
                shape=(self.num_prototypes, self.proto_dim, 1, 1, 1)),
            trainable=True, name="prototype_vectors")
        self.prototype_to_features = layers.Conv3D(
            self.proto_dim, kernel_size=1,
            kernel_initializer="zeros", bias_initializer="zeros",
            name="prototype_to_features")

        self.dec3     = DecoderBlock(c * 4)
        self.dec2     = DecoderBlock(c * 2)
        self.dec1     = DecoderBlock(c)
        self.out_conv = layers.Conv3D(n_classes, 1,
                                      kernel_initializer="glorot_uniform")

    def _l2_distances(self, x):
        proto_filters = tf.transpose(self.prototype_vectors, perm=[2, 3, 4, 1, 0])
        dot = tf.nn.conv3d(x, filters=proto_filters,
                           strides=[1, 1, 1, 1, 1], padding="SAME")
        x2  = tf.reduce_sum(tf.square(x), axis=-1, keepdims=True)
        p2  = tf.reshape(
            tf.reduce_sum(tf.square(self.prototype_vectors), axis=[1, 2, 3, 4]),
            [1, 1, 1, 1, -1])
        return tf.sqrt(tf.maximum(x2 - 2.0 * dot + p2, 1e-8))

    def _similarities(self, distances):
        return tf.math.log((distances + 1.0) / (distances + 1e-4))

    def _proto_bottleneck(self, x):
        distances    = self._l2_distances(x)
        similarities = self._similarities(distances)
        proto_feat   = self.prototype_to_features(similarities)
        return x + proto_feat, similarities, distances

    def _decode(self, x, s1, s2, s3, training):
        x = self.dropout(x, training=training)
        x = self.dec3(x, s3)
        x = self.dec2(x, s2)
        x = self.dec1(x, s1)
        return self.out_conv(x)

    def call(self, inputs, training=False):
        s1, x = self.enc1(inputs)
        s2, x = self.enc2(x)
        s3, x = self.enc3(x)
        x = self.bottleneck(x)
        x, _, _ = self._proto_bottleneck(x)
        return self._decode(x, s1, s2, s3, training)

    def forward_with_similarities(self, inputs):
        s1, x = self.enc1(inputs)
        s2, x = self.enc2(x)
        s3, x = self.enc3(x)
        x = self.bottleneck(x)
        x, sims, _ = self._proto_bottleneck(x)
        return self._decode(x, s1, s2, s3, training=False), sims
'''

with open('/content/unet3d_step3.py', 'w') as f:
    f.write(model_code)

import sys
sys.path.insert(0, '/content')
print('unet3d_step3.py written')

In [ ]:
import tarfile
import numpy as np

s3 = boto3.client('s3')

if JOB_NAME is None:
    resp = s3.list_objects_v2(Bucket=S3_BUCKET, Prefix=S3_OUTPUT_PREFIX + '/', Delimiter='/')
    jobs = sorted([
        p['Prefix'].split('/')[-2]
        for p in resp.get('CommonPrefixes', [])
        if 'unet-proto-losses' in p['Prefix']
    ])
    if not jobs:
        raise RuntimeError(f'No unet-proto-losses jobs found under s3://{S3_BUCKET}/{S3_OUTPUT_PREFIX}/')
    JOB_NAME = jobs[-1]
    print(f'Available jobs: {jobs}')
    print(f'Auto-selected : {JOB_NAME}')

weights_path = os.path.join(MODEL_DIR, 'best_model.weights.h5')

if not os.path.exists(weights_path):
    s3_key  = f'{S3_OUTPUT_PREFIX}/{JOB_NAME}/{JOB_NAME}/output/model.tar.gz'
    tarball = os.path.join(MODEL_DIR, 'model.tar.gz')
    print(f'Downloading s3://{S3_BUCKET}/{s3_key} ...')
    s3.download_file(S3_BUCKET, s3_key, tarball)
    with tarfile.open(tarball) as t:
        t.extractall(MODEL_DIR)
    os.remove(tarball)
    print(f'Extracted: {os.listdir(MODEL_DIR)}')

assert os.path.exists(weights_path), (
    f'best_model.weights.h5 not found in {MODEL_DIR}. '
    f'Files present: {os.listdir(MODEL_DIR)}')
print(f'Weights ready: {weights_path}')

In [ ]:
tf.get_logger().setLevel('ERROR')
from unet3d_step3 import UNet3DProto
import h5py

model = UNet3DProto(n_classes=N_CLASSES,
                    base_channels=BASE_CHANNELS,
                    protos_per_class=PROTOS_PER_CLASS)

dummy = tf.zeros([1, NUM_SLICES, HEIGHT, WIDTH, 4])
_ = model(dummy, training=False)

# ── Explicit h5-path load ─────────────────────────────────────────────────────
# The checkpoint uses attribute names as h5 keys (enc1, dec3, bottleneck, …).
# prototype_to_features and out_conv were stored as layers/conv3d and conv3d_1
# (auto-generated Keras names).  We load each tensor by its exact h5 path so
# neither layer-name mismatches nor the missing prototype_to_features entry
# cause problems.

def _conv_block(blk, g):
    blk.conv1.kernel.assign(g['conv1']['vars']['0'][:])
    blk.norm1.gamma.assign(g['norm1']['vars']['0'][:])
    blk.norm1.beta.assign(  g['norm1']['vars']['1'][:])
    blk.conv2.kernel.assign(g['conv2']['vars']['0'][:])
    blk.norm2.gamma.assign(g['norm2']['vars']['0'][:])
    blk.norm2.beta.assign(  g['norm2']['vars']['1'][:])

def _enc(enc, g):
    _conv_block(enc.conv, g['conv'])

def _dec(dec, g):
    dec.upsample.kernel.assign(g['upsample']['vars']['0'][:])
    dec.upsample.bias.assign(  g['upsample']['vars']['1'][:])
    _conv_block(dec.conv, g['conv'])

with h5py.File(weights_path, 'r') as f:
    model.prototype_vectors.assign(f['vars']['0'][:])

    _enc(model.enc1, f['enc1'])
    _enc(model.enc2, f['enc2'])
    _enc(model.enc3, f['enc3'])
    _conv_block(model.bottleneck, f['bottleneck'])
    _dec(model.dec3, f['dec3'])
    _dec(model.dec2, f['dec2'])
    _dec(model.dec1, f['dec1'])

    # Identify prototype_to_features vs out_conv by kernel input-channel count
    for key in ('conv3d', 'conv3d_1'):
        k = f['layers'][key]['vars']['0'][:]   # kernel tensor
        b = f['layers'][key]['vars']['1'][:]   # bias tensor
        if k.shape[-2] == NUM_PROTOTYPES:      # in_ch == 15 → prototype_to_features
            model.prototype_to_features.kernel.assign(k)
            model.prototype_to_features.bias.assign(b)
        else:                                  # in_ch == BASE_CHANNELS → out_conv
            model.out_conv.kernel.assign(k)
            model.out_conv.bias.assign(b)

proto_norm = float(tf.norm(model.prototype_vectors).numpy())
print(f'Loaded   : {weights_path}')
print(f'Params   : {model.count_params():,}')
print(f'prototype_vectors  shape={model.prototype_vectors.shape}  norm={proto_norm:.4f}')
print(f'enc1.conv.conv1.kernel  shape={model.enc1.conv.conv1.kernel.shape}')
print(f'out_conv.kernel         shape={model.out_conv.kernel.shape}')
print('✓ All weights loaded')

In [ ]:
# ── Debug: only run this if the cell above printed ⚠ ─────────────────────────
# Prints the full h5 structure so you can see what's actually in the checkpoint.

def print_h5(node, indent=0):
    for key in node.keys():
        item = node[key]
        prefix = '  ' * indent
        if isinstance(item, h5py.Group):
            print(f'{prefix}[{key}]')
            print_h5(item, indent + 1)
        else:
            print(f'{prefix}{key}: shape={item.shape}  dtype={item.dtype}')

with h5py.File(weights_path, 'r') as f:
    print_h5(f)

In [ ]:
import h5py
from tqdm.notebook import tqdm

print(f'Downloading volume {VOLUME_ID} ({NUM_SLICES} slices)...')
for s in tqdm(range(NUM_SLICES)):
    fname = f'volume_{VOLUME_ID}_slice_{s}.h5'
    local = os.path.join(DATA_DIR, fname)
    if not os.path.exists(local):
        s3.download_file(S3_BUCKET, f'{S3_DATA_PREFIX}/{fname}', local)

img_slices, mask_slices = [], []
for s in range(NUM_SLICES):
    with h5py.File(os.path.join(DATA_DIR, f'volume_{VOLUME_ID}_slice_{s}.h5'), 'r') as f:
        img_slices.append(f['image'][:].astype(np.float32))
        mask_slices.append(f['mask'][:].astype(np.float32))

vol  = np.stack(img_slices,  axis=0)   # (D, H, W, 4)
mask = np.stack(mask_slices, axis=0)   # (D, H, W, 3)  one-hot tumor classes

# Normalise image globally
vmin, vmax = vol.min(), vol.max()
if vmax - vmin > 1e-8:
    vol = (vol - vmin) / (vmax - vmin)

# Add background channel → (D, H, W, 4)
bg   = (mask.sum(-1, keepdims=True) == 0).astype(np.float32)
mask = np.concatenate([bg, mask], axis=-1)

print(f'vol  shape : {vol.shape}  range [{vol.min():.3f}, {vol.max():.3f}]')
print(f'mask shape : {mask.shape}  unique classes: {np.unique(np.argmax(mask, axis=-1))}')

In [ ]:
from scipy.ndimage import zoom as nd_zoom

print('Running forward pass...')
vol_t           = tf.constant(vol[np.newaxis], dtype=tf.float32)
logits, sims    = model.forward_with_similarities(vol_t)

sims_bn  = sims.numpy()[0]                                      # (D, Hb, Wb, P)
pred_soft = tf.nn.softmax(logits, axis=-1).numpy()[0]           # (D, H, W, 4)
pred_hard = np.argmax(pred_soft, axis=-1)                       # (D, H, W)

# Upsample similarities to full image resolution
sims_up = nd_zoom(sims_bn, (1, HEIGHT / Hb, WIDTH / Wb, 1), order=1)  # (D, H, W, P)

# Per-prototype peak depth
peak_depth = [
    int(sims_up[:, :, :, k].max(axis=(1, 2)).argmax())
    for k in range(NUM_PROTOTYPES)
]

print(f'Similarities : {sims_bn.shape} → upsampled to {sims_up.shape}')
print('\nPeak depth per prototype:')
for k in range(NUM_PROTOTYPES):
    cls = k // PROTOS_PER_CLASS + 1
    print(f'  {CLASS_NAMES[cls]}-p{k % PROTOS_PER_CLASS}: '
          f'depth={peak_depth[k]:3d}  '
          f'max_sim={sims_up[peak_depth[k],:,:,k].max():.4f}')

## Figure 1 — 3-D Contact Sheet

Each row = one prototype.  
Columns = 8 evenly-spaced axial slices through the volume.  
**Red border** marks the slice with the highest prototype activation.  
Coloured contour = ground-truth segmentation for that prototype's class.

In [ ]:
import matplotlib.pyplot as plt
import matplotlib.patches as mpatches

SHOW_DEPTHS = [int(d) for d in np.linspace(4, NUM_SLICES - 5, 8).round()]
N_COLS = 1 + len(SHOW_DEPTHS)

fig, axes = plt.subplots(NUM_PROTOTYPES, N_COLS,
                          figsize=(N_COLS * 2.6, NUM_PROTOTYPES * 2.4))

for k in range(NUM_PROTOTYPES):
    cls   = k // PROTOS_PER_CLASS + 1
    color = CLASS_COLORS[cls]
    pd    = peak_depth[k]
    max_s = float(sims_up[:, :, :, k].max())
    min_s = float(sims_up[:, :, :, k].min())

    axes[k, 0].text(
        0.5, 0.5,
        f'{CLASS_NAMES[cls]}-p{k % PROTOS_PER_CLASS}\n'
        f'peak depth={pd}\n'
        f'max sim={max_s:.3f}',
        ha='center', va='center', fontsize=8.5,
        bbox=dict(boxstyle='round', facecolor='#fff8e7', alpha=0.9))
    axes[k, 0].axis('off')

    for col, d in enumerate(SHOW_DEPTHS):
        ax   = axes[k, col + 1]
        t1ce = vol[d, :, :, 1]
        sim  = sims_up[d, :, :, k]
        gt   = np.argmax(mask[d], axis=-1)

        fg = t1ce[t1ce > 0]
        lo, hi = (np.percentile(fg, [1, 99]) if fg.size > 0 else (0.0, 1.0))
        t1ce_n = np.clip((t1ce - lo) / (hi - lo + 1e-8), 0, 1)

        ax.imshow(t1ce_n, cmap='gray', vmin=0, vmax=1)
        ax.imshow(sim, cmap='hot', alpha=0.40, vmin=min_s, vmax=max_s)
        ax.contour(gt == cls, levels=[0.5], colors=[color], linewidths=0.8)
        ax.set_title(f'd={d}', fontsize=7, pad=1)
        ax.set_xticks([]); ax.set_yticks([])

        if d == pd:
            for spine in ax.spines.values():
                spine.set_edgecolor('red'); spine.set_linewidth(2.5)
        else:
            for spine in ax.spines.values():
                spine.set_edgecolor(color); spine.set_linewidth(0.6)

for cls in [1, 2, 3]:
    row = (cls - 1) * PROTOS_PER_CLASS
    for col in range(N_COLS):
        axes[row, col].spines['top'].set_linewidth(2.0)
        axes[row, col].spines['top'].set_edgecolor('#333333')

plt.suptitle(
    f'Prototype 3-D Contact Sheet — Volume {VOLUME_ID}  |  {PROTOS_PER_CLASS} protos/class\n'
    f'(hot overlay = prototype similarity  |  red border = peak depth  |  '
    f'coloured contour = GT class)',
    fontsize=10, fontweight='bold', y=1.002)

plt.tight_layout()
out = os.path.join(OUTPUT_DIR, f'contact_sheet_vol{VOLUME_ID}.png')
plt.savefig(out, dpi=150, bbox_inches='tight')
plt.show()
print(f'Saved: {out}')

## Figure 2 — Orthogonal View at Peak Activation

For each prototype, shows the axial / coronal / sagittal slice
that passes through the voxel with the highest similarity score.  
Right-most column shows the predicted segmentation on the same axial slice.

In [ ]:
fig, axes = plt.subplots(NUM_PROTOTYPES, 4,
                          figsize=(4 * 3.5, NUM_PROTOTYPES * 3.2))

for k in range(NUM_PROTOTYPES):
    cls   = k // PROTOS_PER_CLASS + 1
    color = CLASS_COLORS[cls]

    flat_idx = int(sims_up[:, :, :, k].argmax())
    pd, ph, pw = np.unravel_index(flat_idx, sims_up[:, :, :, k].shape)

    def t1ce_norm(arr):
        fg = arr[arr > 0]
        lo, hi = (np.percentile(fg, [1, 99]) if fg.size > 0 else (0.0, 1.0))
        return np.clip((arr - lo) / (hi - lo + 1e-8), 0, 1)

    axial_t1ce    = t1ce_norm(vol[pd, :, :, 1])
    coronal_t1ce  = t1ce_norm(vol[:, ph, :, 1])
    sagittal_t1ce = t1ce_norm(vol[:, :, pw, 1])

    axial_sim    = sims_up[pd, :, :, k]
    coronal_sim  = sims_up[:, ph, :, k]
    sagittal_sim = sims_up[:, :, pw, k]

    axial_gt    = np.argmax(mask[pd], axis=-1)
    coronal_gt  = np.argmax(mask[:, ph, :, :], axis=-1)
    sagittal_gt = np.argmax(mask[:, :, pw, :], axis=-1)

    vmin_k = float(sims_up[:, :, :, k].min())
    vmax_k = float(sims_up[:, :, :, k].max())

    panels = [
        (axes[k, 0], axial_t1ce,    axial_sim,    axial_gt,    f'Axial  d={pd}'),
        (axes[k, 1], coronal_t1ce,  coronal_sim,  coronal_gt,  f'Coronal h={ph}'),
        (axes[k, 2], sagittal_t1ce, sagittal_sim, sagittal_gt, f'Sagittal w={pw}'),
    ]
    for ax, img, sim, gt, title in panels:
        ax.imshow(img, cmap='gray', vmin=0, vmax=1)
        ax.imshow(sim, cmap='hot', alpha=0.40, vmin=vmin_k, vmax=vmax_k)
        ax.contour(gt == cls, levels=[0.5], colors=[color], linewidths=1.0)
        ax.set_title(f'{CLASS_NAMES[cls]}-p{k % PROTOS_PER_CLASS}  {title}', fontsize=8)
        ax.set_xticks([]); ax.set_yticks([])
        for spine in ax.spines.values():
            spine.set_edgecolor(color); spine.set_linewidth(1.2)

    ax4 = axes[k, 3]
    ax4.imshow(axial_t1ce, cmap='gray', vmin=0, vmax=1)
    seg_float = pred_hard[pd].astype(float)
    ax4.imshow(np.ma.masked_where(seg_float == 0, seg_float),
               cmap='Set1', vmin=1, vmax=3, alpha=0.45)
    ax4.scatter([pw], [ph], c='red', s=40, marker='+', linewidths=1.5)
    ax4.set_title(f'Pred seg  d={pd}', fontsize=8)
    ax4.set_xticks([]); ax4.set_yticks([])

plt.suptitle(
    f'Prototype Orthogonal Views at Peak Activation — Volume {VOLUME_ID}\n'
    f'(hot overlay = similarity  |  coloured contour = GT  |  "+" = peak voxel)',
    fontsize=10, fontweight='bold', y=1.001)

plt.tight_layout()
out = os.path.join(OUTPUT_DIR, f'ortho_views_vol{VOLUME_ID}.png')
plt.savefig(out, dpi=150, bbox_inches='tight')
plt.show()
print(f'Saved: {out}')